# ABRIR DOCKER DESKTOP Y CONTAINER BLAST NCBI

In [1]:
import subprocess

result = subprocess.run(
    ["docker", "run", "--rm", "ncbi/blast", "blastn", "-version"],
    capture_output=True,
    text=True
)

print(result.stdout)

blastn: 2.17.0+
 Package: blast 2.17.0, build Jul 29 2025 18:15:49



# CREAR BASE DE REFERENCIA 

In [2]:
from pathlib import Path
import subprocess

reference_fasta = Path(
    r"C:\Users\berna\Desktop\META DIV BUILDER V1.6\output\ITS\For_R\only_fungi\species_only_p1_sppn08\sequences.fasta"
)

blast_db_dir = Path(
    r"C:\Users\berna\Desktop\META DIV BUILDER V1.6\BLAST\blast_db"
)

blast_db_dir.mkdir(exist_ok=True)

cmd = [
    "docker", "run", "--rm",
    "-v", f"{reference_fasta.parent}:/data",
    "-v", f"{blast_db_dir}:/blast_db",
    "ncbi/blast",
    "makeblastdb",
    "-in", f"/data/{reference_fasta.name}",
    "-dbtype", "nucl",
    "-out", "/blast_db/metadiv_db"
]

subprocess.run(cmd, check=True)

CompletedProcess(args=['docker', 'run', '--rm', '-v', 'C:\\Users\\berna\\Desktop\\META DIV BUILDER V1.6\\output\\ITS\\For_R\\only_fungi\\species_only_p1_sppn08:/data', '-v', 'C:\\Users\\berna\\Desktop\\META DIV BUILDER V1.6\\BLAST\\blast_db:/blast_db', 'ncbi/blast', 'makeblastdb', '-in', '/data/sequences.fasta', '-dbtype', 'nucl', '-out', '/blast_db/metadiv_db'], returncode=0)

# CORRER BLASTN

In [3]:
from pathlib import Path
import subprocess
import pandas as pd

# Paths
project_dir = Path(r"C:\Users\berna\Desktop\META DIV BUILDER V1.6")
blast_dir = project_dir / "BLAST"

query_dir = blast_dir / "query"
db_dir = blast_dir / "blast_db"
results_dir = blast_dir / "blast_results"

results_dir.mkdir(parents=True, exist_ok=True)

# Detect query FASTA automatically
query_files = list(query_dir.glob("*.fasta")) + list(query_dir.glob("*.fa"))

if len(query_files) == 0:
    raise FileNotFoundError("No FASTA file found in BLAST/query")

if len(query_files) > 1:
    raise ValueError("More than one FASTA file found in BLAST/query. Keep only one.")

query_fasta = query_files[0]

# Output
blast_tsv = results_dir / "blast_results.tsv"

# Run BLAST
cmd = [
    "docker", "run", "--rm",
    "-v", f"{query_dir}:/query",
    "-v", f"{db_dir}:/blast_db",
    "-v", f"{results_dir}:/results",
    "ncbi/blast",
    "blastn",
    "-query", f"/query/{query_fasta.name}",
    "-db", "/blast_db/metadiv_db",
    "-out", "/results/blast_results.tsv",
    "-outfmt",
    "6 qseqid sseqid pident length mismatch gapopen qstart qend sstart send evalue bitscore qcovs",
    "-evalue", "1e-20",
    "-num_threads", "8",
    "-max_target_seqs", "10"
]

print("Running BLAST...")
subprocess.run(cmd, check=True)
print("BLAST completed.")

# Load results
cols = [
    "qseqid", "sseqid", "pident", "length", "mismatch", "gapopen",
    "qstart", "qend", "sstart", "send", "evalue", "bitscore", "qcovs"
]

blast_df = pd.read_csv(
    blast_tsv,
    sep="\t",
    header=None,
    names=cols
)

# Save all hits
blast_df.to_csv(results_dir / "blast_all_hits.csv", index=False)

# Best hit per query
best_hits = (
    blast_df
    .sort_values(
        by=["qseqid", "bitscore", "qcovs", "pident", "evalue"],
        ascending=[True, False, False, False, True]
    )
    .drop_duplicates("qseqid")
    .reset_index(drop=True)
)

best_hits.to_csv(results_dir / "blast_best_hits.csv", index=False)

print("Total hits:", len(blast_df))
print("Queries with hits:", best_hits["qseqid"].nunique())
print("Results saved in:", results_dir)

best_hits

Running BLAST...
BLAST completed.
Total hits: 191
Queries with hits: 19
Results saved in: C:\Users\berna\Desktop\META DIV BUILDER V1.6\BLAST\blast_results


,qseqid,sseqid,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qcovs
0,AY061655_Russula_amoenicolor_Seq_US,Russula_0662,89.389,622,32,15,41,630,1,620,0.000000e+00,767,87
1,EU019934_Russula_variispora_Seq_AU,Russula_0662,88.719,523,32,15,1,497,27,548,6.110000e-176,617,36
2,EU019938_Russula_rostraticystidia_Seq_AU,Russula_0662,90.711,506,27,12,11,500,47,548,0.000000e+00,660,35
3,KT824241_Russula_intervenosa_RefTypeSeq_IN,Russula_0662,89.184,564,35,15,1,540,50,611,0.000000e+00,684,100
4,KX234819_Russula_pseudoamoenicolor_RepSeq_IN,Russula_0662,97.092,619,10,4,1,613,4,620,0.000000e+00,1040,97
5,MF535185_Russula_pauriensis_RepSeq_IN,Russula_0662,95.980,597,16,4,1,591,26,620,0.000000e+00,966,89
6,MK105627_Russula_andaluciana_ReqSeq_ES,Russula_0662,91.284,631,26,15,43,655,1,620,0.000000e+00,837,93
7,MN658517_Russula_paravioleipes_RefHolotypeSeq_CN,Russula_0662,92.097,620,28,8,11,611,3,620,0.000000e+00,857,91
8,MT017548_Russula_orientipurpurea_TypeSeq_KR,Russula_0662,90.943,541,34,6,3,529,11,550,0.000000e+00,717,95
9,MT537255_Russula_diversicolor_Seq_MQ,Russula_0748,86.474,658,44,23,46,690,1,626,0.000000e+00,680,88
